# Inference Pipeline & Deployment Preparation

## Objective

The purpose of this notebook is to build a production-ready prediction pipeline using the trained machine learning model.

Unlike the training notebooks, this notebook does not train any model. Instead, it focuses on using the saved preprocessing pipeline and trained model to generate predictions for new flight data.

This notebook covers:

- Loading saved artifacts
- Creating a reusable prediction pipeline
- Single prediction
- Batch prediction
- Exporting prediction results
- Deployment preparation

This notebook represents the inference stage of the machine learning lifecycle.

In [1]:
# =====================================================
# Import Libraries
# =====================================================

import joblib
import pandas as pd
import numpy as np

print("Libraries Imported Successfully.")

Libraries Imported Successfully.


# Load Saved Artifacts

## Objective

In this step, we load the saved machine learning artifacts required for inference.

The artifacts include:

- **Preprocessor** – transforms raw input data into the format expected by the model.
- **Best Model** – generates flight ticket price predictions.

Loading these artifacts ensures that the prediction pipeline uses the exact preprocessing steps and trained model from the training phase.

In [3]:
# =====================================================
# Load Saved Artifacts
# =====================================================

import os
import joblib

# Define artifact paths
PREPROCESSOR_PATH = "artifacts/preprocessor.pkl"
MODEL_PATH = "artifacts/best_model.pkl"

# Check if files exist
if not os.path.exists(PREPROCESSOR_PATH):
    raise FileNotFoundError(f"Preprocessor not found: {PREPROCESSOR_PATH}")

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"Model not found: {MODEL_PATH}")

# Load artifacts
preprocessor = joblib.load(PREPROCESSOR_PATH)
best_model = joblib.load(MODEL_PATH)

print("=" * 60)
print("Artifacts Loaded Successfully")
print("=" * 60)

print(f"Preprocessor Type : {type(preprocessor).__name__}")
print(f"Model Type        : {type(best_model).__name__}")

Artifacts Loaded Successfully
Preprocessor Type : ColumnTransformer
Model Type        : XGBRegressor


# Create Reusable Prediction Function

## Objective

To build a reusable prediction function that can be used across different applications.

The function performs the complete inference pipeline:

1. Accept raw input data
2. Convert input into a DataFrame
3. Apply the saved preprocessing pipeline
4. Generate prediction using the trained model
5. Return the predicted flight ticket price

This function represents the core inference logic of the project and can be reused during deployment.

In [4]:
# =====================================================
# Reusable Prediction Function
# =====================================================

def predict_flight_price(input_data):
    """
    Predict flight ticket price using the trained model.

    Parameters
    ----------
    input_data : dict or pandas.DataFrame
        Raw flight information.

    Returns
    -------
    float
        Predicted flight ticket price.
    """

    # Convert dictionary to DataFrame
    if isinstance(input_data, dict):
        input_df = pd.DataFrame([input_data])

    elif isinstance(input_data, pd.DataFrame):
        input_df = input_data.copy()

    else:
        raise ValueError(
            "Input must be either a dictionary or a pandas DataFrame."
        )

    # Apply preprocessing
    processed_data = preprocessor.transform(input_df)

    # Generate prediction
    prediction = best_model.predict(processed_data)

    # Return prediction
    return float(prediction[0])


print("Prediction function created successfully.")

Prediction function created successfully.


# Single Flight Price Prediction

## Objective

This step demonstrates how to predict the ticket price for a single flight using the trained machine learning model.

The process includes:

- Creating a sample flight record
- Passing it to the prediction function
- Displaying the estimated ticket price

This simulates how a real-world application predicts the price for one user request.

In [5]:
# =====================================================
# Single Flight Price Prediction
# =====================================================

# Sample Flight Data
sample_flight = {
    "airline": "Indigo",
    "source_city": "Delhi",
    "departure_time": "Morning",
    "stops": "one",
    "arrival_time": "Evening",
    "destination_city": "Mumbai",
    "class": "Economy",
    "duration": 2.17,
    "days_left": 15
}

# Predict Price
predicted_price = predict_flight_price(sample_flight)

print("=" * 60)
print("Single Flight Prediction")
print("=" * 60)

print(f"Predicted Flight Ticket Price: ₹ {predicted_price:,.2f}")

Single Flight Prediction
Predicted Flight Ticket Price: ₹ 4,107.05


# Batch Flight Price Prediction

## Objective

This step demonstrates batch inference using multiple flight records from the cleaned dataset.

Instead of predicting a single flight, we generate predictions for a batch of records and append the predicted prices to the dataset.

Batch prediction is commonly used in production systems for analytics, reporting, and large-scale inference.

In [7]:
# =====================================================
# Batch Flight Price Prediction
# =====================================================

# Load cleaned dataset
batch_df = pd.read_csv("Dataset/clean_Dataset.csv")

# Remove target column (price) if present
if "price" in batch_df.columns:
    batch_input = batch_df.drop(columns=["price"])
else:
    batch_input = batch_df.copy()

# Take first 10 records for demonstration
batch_input = batch_input.head(10)

# Apply preprocessing
batch_processed = preprocessor.transform(batch_input)

# Generate predictions
batch_predictions = best_model.predict(batch_processed)

# Add predictions
batch_result = batch_input.copy()
batch_result["Predicted_Price"] = batch_predictions.round(2)

print("=" * 60)
print("Batch Prediction Completed Successfully")
print("=" * 60)

display(batch_result)

Batch Prediction Completed Successfully


,Unnamed: 0,airline,flight,source_city,departure_time,stops,arrival_time,destination_city,class,duration,days_left,Predicted_Price
0,0,SpiceJet,SG-8709,Delhi,Evening,zero,Night,Mumbai,Economy,2.17,1,9014.780273
1,1,SpiceJet,SG-8157,Delhi,Early_Morning,zero,Morning,Mumbai,Economy,2.33,1,6344.890137
2,2,AirAsia,I5-764,Delhi,Early_Morning,zero,Early_Morning,Mumbai,Economy,2.17,1,5640.399902
3,3,Vistara,UK-995,Delhi,Morning,zero,Afternoon,Mumbai,Economy,2.25,1,7070.120117
4,4,Vistara,UK-963,Delhi,Morning,zero,Morning,Mumbai,Economy,2.33,1,6805.319824
5,5,Vistara,UK-945,Delhi,Morning,zero,Afternoon,Mumbai,Economy,2.33,1,7162.259766
6,6,Vistara,UK-927,Delhi,Morning,zero,Morning,Mumbai,Economy,2.08,1,6878.609863
7,7,Vistara,UK-951,Delhi,Afternoon,zero,Evening,Mumbai,Economy,2.17,1,7215.859863
8,8,GO_FIRST,G8-334,Delhi,Early_Morning,zero,Morning,Mumbai,Economy,2.17,1,6114.220215
9,9,GO_FIRST,G8-336,Delhi,Afternoon,zero,Evening,Mumbai,Economy,2.25,1,6619.660156


# Save Prediction Results

## Objective

After generating predictions, the results should be saved for further analysis and reporting.

Saving prediction results enables:

- Business reporting
- Dashboard integration
- Data sharing
- Future auditing
- Batch inference pipelines

The output file contains both the input flight details and the predicted ticket prices.

In [9]:
# =====================================================
# Save Prediction Results
# =====================================================

import os

# Create output directory if it doesn't exist
os.makedirs("outputs", exist_ok=True)

# Output file path
OUTPUT_PATH = "outputs/flight_price_predictions.csv"

# Save predictions
batch_result.to_csv(OUTPUT_PATH, index=False)

print("=" * 60)
print("Prediction Results Saved Successfully")
print("=" * 60)

print(f"Output File : {OUTPUT_PATH}")
print(f"Total Records Saved : {len(batch_result)}")

# Display first few rows
display(batch_result.head())

Prediction Results Saved Successfully
Output File : outputs/flight_price_predictions.csv
Total Records Saved : 10


,Unnamed: 0,airline,flight,source_city,departure_time,stops,arrival_time,destination_city,class,duration,days_left,Predicted_Price
0,0,SpiceJet,SG-8709,Delhi,Evening,zero,Night,Mumbai,Economy,2.17,1,9014.780273
1,1,SpiceJet,SG-8157,Delhi,Early_Morning,zero,Morning,Mumbai,Economy,2.33,1,6344.890137
2,2,AirAsia,I5-764,Delhi,Early_Morning,zero,Early_Morning,Mumbai,Economy,2.17,1,5640.399902
3,3,Vistara,UK-995,Delhi,Morning,zero,Afternoon,Mumbai,Economy,2.25,1,7070.120117
4,4,Vistara,UK-963,Delhi,Morning,zero,Morning,Mumbai,Economy,2.33,1,6805.319824


# Input Validation & Error Handling

## Objective

Before making predictions, it is important to validate the input data.

This step ensures:

- All required columns are present
- Missing values are detected
- Invalid inputs are identified
- Clear error messages are provided

Input validation improves the reliability and robustness of the prediction pipeline.

In [10]:
# =====================================================
# Input Validation Function
# =====================================================

# Required input features
REQUIRED_COLUMNS = [
    "airline",
    "source_city",
    "departure_time",
    "stops",
    "arrival_time",
    "destination_city",
    "class",
    "duration",
    "days_left"
]


def validate_input(input_df):
    """
    Validate input data before prediction.

    Parameters
    ----------
    input_df : pandas.DataFrame

    Returns
    -------
    bool
    """

    # Check missing columns
    missing_columns = [
        col for col in REQUIRED_COLUMNS
        if col not in input_df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Missing Columns: {missing_columns}"
        )

    # Check missing values
    if input_df[REQUIRED_COLUMNS].isnull().sum().sum() > 0:
        raise ValueError(
            "Input data contains missing values."
        )

    return True

In [11]:
# =====================================================
# Validate Batch Data
# =====================================================

try:

    validate_input(batch_input)

    print("=" * 60)
    print("Input Validation Passed")
    print("=" * 60)

except Exception as e:

    print("=" * 60)
    print("Validation Failed")
    print("=" * 60)

    print(e)

Input Validation Passed


# End-to-End Pipeline Testing

## Objective

Before deploying a machine learning model, the complete inference pipeline must be tested.

This test verifies that:

- Input validation works correctly
- Data preprocessing is applied successfully
- The trained model generates predictions
- The pipeline executes without errors

A successful end-to-end test confirms that the prediction pipeline is ready for deployment.

In [12]:
# =====================================================
# End-to-End Pipeline Testing
# =====================================================

try:

    # Step 1 : Validate Input
    validate_input(batch_input)

    # Step 2 : Generate Predictions
    predictions = best_model.predict(
        preprocessor.transform(batch_input)
    )

    # Step 3 : Prepare Output
    test_result = batch_input.copy()
    test_result["Predicted_Price"] = predictions.round(2)

    print("=" * 70)
    print("End-to-End Pipeline Test Passed")
    print("=" * 70)

    print(f"Total Records Tested : {len(test_result)}")
    print(f"Prediction Status    : Successful")

    display(test_result.head())

except Exception as e:

    print("=" * 70)
    print("Pipeline Test Failed")
    print("=" * 70)

    print(f"Error : {e}")

End-to-End Pipeline Test Passed
Total Records Tested : 10
Prediction Status    : Successful


,Unnamed: 0,airline,flight,source_city,departure_time,stops,arrival_time,destination_city,class,duration,days_left,Predicted_Price
0,0,SpiceJet,SG-8709,Delhi,Evening,zero,Night,Mumbai,Economy,2.17,1,9014.780273
1,1,SpiceJet,SG-8157,Delhi,Early_Morning,zero,Morning,Mumbai,Economy,2.33,1,6344.890137
2,2,AirAsia,I5-764,Delhi,Early_Morning,zero,Early_Morning,Mumbai,Economy,2.17,1,5640.399902
3,3,Vistara,UK-995,Delhi,Morning,zero,Afternoon,Mumbai,Economy,2.25,1,7070.120117
4,4,Vistara,UK-963,Delhi,Morning,zero,Morning,Mumbai,Economy,2.33,1,6805.319824


# Deployment Readiness Checklist

## Objective

Before deploying a machine learning model into production, it is important to verify that all critical components are available and functioning correctly.

This checklist helps ensure that:

- The trained model is available
- The preprocessing pipeline is saved
- The prediction pipeline works correctly
- Predictions can be exported
- The project is ready for deployment

A deployment checklist is a common practice in industry to reduce deployment risks and improve reliability.

In [13]:
# =====================================================
# Deployment Readiness Checklist
# =====================================================

deployment_checklist = {
    "Preprocessor Loaded": preprocessor is not None,
    "Model Loaded": best_model is not None,
    "Prediction Function Created": callable(predict_flight_price),
    "Input Validation Available": callable(validate_input),
    "Batch Prediction Completed": "batch_result" in globals(),
    "Prediction File Saved": os.path.exists(OUTPUT_PATH),
}

print("=" * 70)
print("Deployment Readiness Checklist")
print("=" * 70)

all_checks_passed = True

for item, status in deployment_checklist.items():

    icon = "✅" if status else "❌"

    print(f"{icon} {item}")

    if not status:
        all_checks_passed = False

print("=" * 70)

if all_checks_passed:
    print("🎉 Project Status : READY FOR DEPLOYMENT")
else:
    print("⚠️ Project Status : NOT READY")
    print("Please resolve the failed checks before deployment.")

Deployment Readiness Checklist
✅ Preprocessor Loaded
✅ Model Loaded
✅ Prediction Function Created
✅ Input Validation Available
✅ Batch Prediction Completed
✅ Prediction File Saved
🎉 Project Status : READY FOR DEPLOYMENT


# Final Summary & Deployment Roadmap

## Objective

This notebook completed the production inference pipeline for the Flight Ticket Price Prediction project.

The following components were successfully implemented:

- Loading trained artifacts
- Prediction pipeline
- Single prediction
- Batch prediction
- Input validation
- Error handling
- Exporting prediction results
- End-to-end pipeline testing
- Deployment readiness verification

This notebook prepares the project for deployment using frameworks such as Streamlit, FastAPI, Flask, or cloud platforms.

In [14]:
# =====================================================
# Final Summary
# =====================================================

print("=" * 75)
print(" Flight Ticket Price Prediction")
print(" Inference Pipeline Completed Successfully")
print("=" * 75)

print("\nProject Components Completed")

completed_steps = [
    "Load Saved Artifacts",
    "Reusable Prediction Function",
    "Single Flight Prediction",
    "Batch Prediction",
    "Prediction Result Export",
    "Input Validation",
    "Error Handling",
    "End-to-End Pipeline Testing",
    "Deployment Readiness Verification"
]

for step in completed_steps:
    print(f"✔ {step}")

print("\nDeployment Readiness")

print("✔ Model (.pkl) Available")
print("✔ Preprocessor (.pkl) Available")
print("✔ Prediction Pipeline Working")
print("✔ Batch Inference Supported")
print("✔ CSV Export Supported")
print("✔ Validation Implemented")

print("\nRecommended Next Steps")

next_steps = [
    "Develop Streamlit Web Application",
    "Build FastAPI REST API",
    "Containerize using Docker",
    "Deploy on AWS / Azure / Render",
    "Implement Logging & Monitoring",
    "Add Model Versioning",
    "Create CI/CD Pipeline"
]

for step in next_steps:
    print(f"➡ {step}")

print("\nOverall Status")

print("🎉 Production Inference Pipeline Successfully Completed")
print("🚀 Project Ready for Application Development & Deployment")

print("=" * 75)

 Flight Ticket Price Prediction
 Inference Pipeline Completed Successfully

Project Components Completed
✔ Load Saved Artifacts
✔ Reusable Prediction Function
✔ Single Flight Prediction
✔ Batch Prediction
✔ Prediction Result Export
✔ Input Validation
✔ Error Handling
✔ End-to-End Pipeline Testing
✔ Deployment Readiness Verification

Deployment Readiness
✔ Model (.pkl) Available
✔ Preprocessor (.pkl) Available
✔ Prediction Pipeline Working
✔ Batch Inference Supported
✔ CSV Export Supported
✔ Validation Implemented

Recommended Next Steps
➡ Develop Streamlit Web Application
➡ Build FastAPI REST API
➡ Containerize using Docker
➡ Deploy on AWS / Azure / Render
➡ Implement Logging & Monitoring
➡ Add Model Versioning
➡ Create CI/CD Pipeline

Overall Status
🎉 Production Inference Pipeline Successfully Completed
🚀 Project Ready for Application Development & Deployment
